# Classifying the type of measure behind each annex
Parses the articles of every consolidated version of Regulation 833/2014 and tags each (article, paragraph, annex) with the type of measure it imposes (export ban, import ban, transit ban, no-re-export clause, price cap, authorisation requirement), keeping the legal text behind each tag. The last cell is an audit that surfaces unclassified rows for manual review. Written for Google Colab.

In [ ]:
# Run once per fresh Colab session.
!pip install -q eurlex pandas beautifulsoup4 lxml


In [ ]:
"""
parse_annex_directions.py
=========================

Parse every consolidated version of Council Regulation (EU) No 833/2014
(Russia sanctions) and extract, for each (article, paragraph), which
annex(es) it cites and which type of trade-restrictive measure it
imposes.

Output: a single long-format CSV that can be joined with a monthly HS6
panel (produced separately by an R pipeline) to yield a direction-aware
panel ready for matching with bilateral trade data.

USAGE
-----
    pip install eurlex pandas beautifulsoup4 lxml
    python parse_annex_directions.py

The script processes all 34 versions of the regulation by default. The
list is hard-coded near the top (ALL_VERSIONS); edit it to run on a
different country's sanctions regulation (e.g. Reg. 765/2006 for Belarus)
or on a subset of versions for debugging.

The output CSV name embeds a timestamp so successive runs do not
overwrite each other.

CHANGES IN v6 (2026-05-20, audit-driven)
----------------------------------------
Discovery audit on the v5 output (Stage 2a, annexes containing goods
codes) surfaced three real misses out of 13 OTHER rows. Patches:

A. New TRANSIT_BAN pattern (`\btransit\b[^.]{0,100}\bvia\b[^.]{0,50}\bRussia\b`).
   I had wrongly assumed Russia had no transit-ban article. Articles
   2a ¶1a (Annex VII), 3c ¶1a (Annex XI / Annex XX) and 3k ¶1a
   (Annex XXXVII) all carry the canonical phrasing "The transit via the
   territory of Russia of goods... shall be prohibited". Same regex
   shape as the Belarus parser's TRANSIT_BAN, with country name
   substituted. TRADE_RELEVANT registered as False (transit isn't a
   bilateral EU↔Russia flow), matching Belarus convention.

B. ANCILLARY "to transport" slack widened from 200 → 300 chars. Audit
   surfaced Article 3n ¶4 (ship-to-ship transfer ban on Russian crude
   oil to third countries) where "transport" and "Russia" are 255 chars
   apart. Old pattern missed.

C. New EXPORT pattern for "sell, or otherwise transfer ownership...
   in Russia". Article 3q ¶1 (sale of tanker vessels, ex 8901 20) is
   functionally an export ban but uses a different verb cluster from
   the canonical "sell, supply, transfer or export". Added alongside;
   mutually exclusive with existing EXPORT patterns.

Remaining OTHER rows (Stage 2a) intentionally NOT patched:
  - Article 3c ¶3 (aircraft repair services): same borderline case as
    Belarus Article 1sa ¶3, deliberately left as OTHER for consistency.
  - Article 12ga ¶1 (IP licensing anti-circumvention): related to
    NO_RE_EXPORT but distinctly different verb cluster; documenting as
    known borderline.
  - Article 3nb ¶1 (temporary storage / free zone of Russian crude):
    storage prohibition with no analogous direction tag; STORAGE_BAN
    would be its own category, deferred.
  - Article 12f ¶3: scope-defining paragraph, correctly OTHER.

CHANGES IN v5 (2026-05-20)
--------------------------
Five fixes backported from the Belarus port, documented at length in
RU-TECHNICAL_notes_Python_parser_patched.txt. In brief:

1. PARA_HEADER_RE now accepts lettered paragraph identifiers (1a, 2aa, ...).
   The old (\\d+) pattern absorbed lettered paragraphs into the preceding
   numbered one, mixing their annex citations and direction verbs.

2. Derogation regex "shall not grant authorisation" widened to also catch
   "shall not grant an authorisation" — a common phrasing missed before.

3. AUTHORISATION_REQUIRED_EXPORT widened: the sale/supply/transfer prefix
   is now optional, so "...authorisation shall be required for the export
   of goods..." also matches.

4. New EXPORT pattern "sell, supply, transfer, export or provide" added
   alongside the original verb cluster.

5. New IMPORT pattern "import, purchase or transfer" added alongside the
   original "purchase, import or transfer". Defensive — catches future
   amendments that might reorder the verb list.

Plus cosmetic: EUR-Lex amendment markers (▼M44, ▼M57, ...) are now
stripped in html_to_plain() to clean up `evidence_for_direction` output.

Output filename also renamed: annex_directions_* → RU-annex_directions_*
for consistency with the rest of the RU pipeline.

DESIGN OVERVIEW
---------------
For each law version:

  1. Fetch the consolidated HTML text via the `eurlex` package.
  2. Convert to plain text, truncating at the first "ANNEX" header so
     the annex tables don't pollute article parsing.
  3. Split into articles using "Article N[a-z]*" headers.
  4. Within each article, split into numbered paragraphs.
  5. Classify each paragraph's legal role:
       - derogation              → SKIPPED
       - prohibition             → KEPT
       - obligation              → KEPT
       - authorisation_required  → KEPT
       - other                   → SKIPPED
  6. For kept paragraphs, extract:
       - annex citations (both internal and external)
       - one or more "direction" matches (EXPORT, IMPORT, PRICE_CAP, ...)
  7. Emit one output row per (annex × direction) combination.

DIRECTION TAXONOMY
------------------
Trade-relevant (these flow into the trade panel):
    EXPORT                              outright EU → Russia goods ban
    IMPORT                              outright Russia → EU goods ban

Documented but deferred (trade_relevant = FALSE):
    AUTHORISATION_REQUIRED_EXPORT       export authorisation regime
    AUTHORISATION_REQUIRED_IMPORT       import authorisation regime (never observed)
    AUTHORISATION_REQUIRED_ANCILLARY    services-side authorisation regime
    PRICE_CAP                           Russian-oil price-cap mechanism
    NO_RE_EXPORT                        anti-circumvention via third countries
    ANCILLARY                           services / financing / insurance / transport
    TRANSACTION_BAN                     entity-level prohibitions
    OTHER                               unclassified prohibitions
"""

from __future__ import annotations

import re
import time
import warnings
from dataclasses import dataclass
from datetime import datetime
from typing import Iterable

import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from eurlex import get_html_by_celex_id

# The eurlex package sometimes hands lxml an XML payload; silence the noisy
# (and harmless for our purposes) warning.
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)


# =============================================================================
# CONFIGURATION
# =============================================================================

# CELEX identifiers for the 34 versions of Reg. 833/2014.
#
# The first entry "32014R0833" is the original 31 July 2014 enactment.
# The other 33 entries are consolidated-text snapshots at the date of each
# amendment, in the form "02014R0833-YYYYMMDD".
#
# To run for a different sanctions regulation (e.g. Belarus = Reg. 765/2006),
# replace this list with the equivalent CELEX IDs. Everything else in this
# file is regulation-agnostic.
ALL_VERSIONS: list[str] = [
    "32014R0833",
    "02014R0833-20140912",
    "02014R0833-20141206",
    "02014R0833-20151009",
    "02014R0833-20171202",
    "02014R0833-20190709",
    "02014R0833-20220228",
    "02014R0833-20220316",
    "02014R0833-20220413",
    "02014R0833-20220604",
    "02014R0833-20220722",
    "02014R0833-20221007",
    "02014R0833-20221204",
    "02014R0833-20221217",
    "02014R0833-20230201",
    "02014R0833-20230205",
    "02014R0833-20230226",
    "02014R0833-20230427",
    "02014R0833-20230624",
    "02014R0833-20231001",
    "02014R0833-20231219",
    "02014R0833-20240215",
    "02014R0833-20240224",
    "02014R0833-20240625",
    "02014R0833-20240913",
    "02014R0833-20241029",
    "02014R0833-20241217",
    "02014R0833-20250225",
    "02014R0833-20250521",
    "02014R0833-20250720",
    "02014R0833-20251024",
    "02014R0833-20251219",
    "02014R0833-20260116",
    "02014R0833-20260424",
]

# Output file name: timestamped so reruns don't overwrite.
OUTPUT_CSV: str = f"RU-annex_directions_all_versions_{datetime.now():%Y-%m-%d_%H-%M}.csv"

# Sibling output introduced 2026-05-28: every annex citation in a paragraph the
# classifier does NOT emit as EXPORT or IMPORT is recorded here, with full
# provenance. See GENERAL-Pipeline_overview_2026-05-28 §"set-aside principle"
# and RU-TECHNICAL_notes_Python_Imp_Exp_Classifier_2026-05-28 §11 (D.21-D.24).
SETASIDE_OUTPUT_CSV: str = f"RU-annex_set_aside_all_versions_{datetime.now():%Y-%m-%d_%H-%M}.csv"

# Width of the contextual snippet in the set-aside CSV, in characters.
SETASIDE_SNIPPET_WIDTH: int = 150

# Sentinel value placed in the `annex` column when a paragraph cites only
# annexes of OTHER regulations (e.g. firearms in "Annex I to Reg. 258/2012")
# and no internal annex of Reg. 833/2014.
NO_INTERNAL_ANNEX_SENTINEL: str = "NO_INTERNAL_ANNEX"


# =============================================================================
# DIRECTION PATTERNS
# =============================================================================
#
# Each entry: (direction_tag, pattern_label, compiled_regex).
#
# A single paragraph may match multiple direction patterns. Each match
# emits a separate output row, so e.g. Article 3g (which combines an
# import ban on goods with related ancillary services in one paragraph)
# emits both IMPORT and ANCILLARY rows.
#
# To extend the parser to catch a new legal mechanism, add a new tuple
# here AND update the TRADE_RELEVANT map below. No other code changes
# are needed.
# -----------------------------------------------------------------------------

DIRECTION_PATTERNS: list[tuple[str, str, re.Pattern]] = [
    # ---- EXPORT (outright EU → Russia goods ban) -------------------------
    ("EXPORT", "sell-supply-transfer-export",
     re.compile(r"\bsell,?\s*supply,?\s*transfer,?\s*or\s*export\b", re.I)),
    # [2026-05-20] Belarus port also has "sell, supply, transfer, export or
    # provide" (extra "or provide" shifts "or" past "export"). Russia may use
    # the same phrasing in software-ban articles added 2022+; pattern added
    # defensively. Mutually exclusive with the form above ("or" position
    # differs), so adding alongside is safe — no double-emit.
    ("EXPORT", "sell-supply-transfer-export-or-provide",
     re.compile(r"\bsell,?\s*supply,?\s*transfer,?\s*export,?\s+or\s+provide\b", re.I)),
    # [2026-05-20 from audit] Article 3q ¶1 (tanker vessel sale ban,
    # ex 8901 20 in Annex XXV): "It shall be prohibited... to sell, or
    # otherwise transfer ownership, directly or indirectly, of tanker
    # vessels... to any natural or legal person, entity or body in Russia
    # or for use in Russia."  The first EXPORT pattern requires the
    # "sell, supply, transfer or export" verb cluster; this paragraph
    # uses a different verb pair ("sell, or otherwise transfer
    # ownership") that the original pattern misses. Adding a separate
    # pattern that anchors on "transfer ownership" + recipient in Russia
    # within 250 chars. Mutually exclusive with the form above (verbs
    # differ), so safe to add alongside.
    ("EXPORT", "sell or transfer ownership ... in Russia",
     re.compile(r"\bsell,?\s*or\s+otherwise\s+transfer\s+ownership\b"
                r"[^.]{0,250}\bin\s+Russia\b", re.I)),

    # ---- IMPORT (outright Russia → EU goods ban) -------------------------
    ("IMPORT", "to import...into the Union",
     re.compile(r"\bto\s+import\b[^.]{0,200}\binto\s+the\s+Union\b", re.I)),
    ("IMPORT", "to purchase...located/originated in Russia",
     re.compile(r"\bto\s+purchase\b[^.]{0,200}\b(located|originated)\s+in\s+Russia\b", re.I)),
    ("IMPORT", "purchase-import-transfer (originate/exported from Russia)",
     re.compile(r"\bto\s+purchase,?\s*import,?\s*(or\s+)?transfer\b", re.I)),
    # [2026-05-20] Belarus port reverses the order to "import, purchase or
    # transfer" (Article 1i ¶1, potash / Annex VIII). Russia's drafting has
    # been consistent so far, but adding this defensively for future
    # amendments. Mutually exclusive with the form above (sequence differs),
    # so adding alongside is safe — no double-emit.
    ("IMPORT", "import-purchase-transfer (reversed order)",
     re.compile(r"\bto\s+import,?\s*purchase,?\s*(or\s+)?transfer\b", re.I)),

    # ---- TRANSIT_BAN [added 2026-05-20 from audit] ----------------------
    # Articles 2a ¶1a (Annex VII), 3c ¶1a (Annex XI / Annex XX), 3k ¶1a
    # (Annex XXXVII): "The transit via the territory of Russia of the
    # goods and technology as listed in Annex N, exported from the Union,
    # shall be prohibited."
    #
    # I had wrongly assumed Russia had no transit-ban article. The audit
    # surfaced these as OTHER+prohibition on goods annexes. Same regex
    # shape as the Belarus parser's TRANSIT_BAN (slack 100/50), country
    # name substituted.
    ("TRANSIT_BAN", "transit ... via ... Russia",
     re.compile(r"\btransit\b[^.]{0,100}\bvia\b[^.]{0,50}\bRussia\b", re.I)),

    # ---- AUTHORISATION REQUIRED (conditional restriction via authorisation regime)
    # The original July 2014 regulation used this mechanism for Annex II
    # (dual-use); it was upgraded to an outright EXPORT ban in March 2022.
    # [2026-05-20] Widened: the verb list (sale, supply, transfer) is now
    # optional. The original verb-list-required form still matches; the
    # shortened "...for the export of goods..." form (found on Belarus
    # Article 1f ¶1aa) now also matches. The broader pattern strictly
    # contains the narrow one's matches, so it replaces rather than
    # adds-alongside (would otherwise double-emit).
    ("AUTHORISATION_REQUIRED_EXPORT", "authorisation required for (sale/supply/transfer or) export",
     re.compile(r"\bauthorisation\s+shall\s+be\s+required\s+for\s+(?:the\s+)?"
                r"(?:sale,?\s*supply,?\s*transfer,?\s*or\s*)?export\b", re.I)),
    ("AUTHORISATION_REQUIRED_IMPORT", "authorisation required for import/purchase",
     re.compile(r"\bauthorisation\s+shall\s+be\s+required\s+for\s+(?:the\s+)?(import|purchase)\b", re.I)),
    ("AUTHORISATION_REQUIRED_ANCILLARY", "shall be subject to authorisation",
     re.compile(r"\bshall\s+be\s+subject\s+to\s+(an\s+)?authorisation\b", re.I)),

    # ---- PRICE_CAP (Russian crude oil / petroleum products, Article 3n)
    ("PRICE_CAP", "price laid down in Annex XXVIII",
     re.compile(r"\bprice\s+laid\s+down\s+in\s+Annex\s+XXVIII\b", re.I)),
    ("PRICE_CAP", "price cap (mechanism / coalition / etc.)",
     re.compile(r"\bprice\s+cap\b", re.I)),

    # ---- NO_RE_EXPORT (contractual anti-circumvention, Article 12g)
    ("NO_RE_EXPORT", "contractually prohibit re-exportation",
     re.compile(r"\bcontractually\s+prohibit\s+re-?exportation\b", re.I)),
    ("NO_RE_EXPORT", "re-exportation to Russia",
     re.compile(r"\bre-?exportation\s+to\s+Russia\b", re.I)),
    ("NO_RE_EXPORT", "ensure subsidiaries comply",
     re.compile(r"\bensure\s+that\s+any\s+legal\s+person\b[^.]{0,150}\bthat\s+they\s+own\s+or\s+control\b", re.I)),

    # ---- ANCILLARY (services, financing, insurance, transport)
    # Travels with goods bans but isn't itself a goods movement.
    ("ANCILLARY", "technical assistance / brokering / other services",
     re.compile(r"\bprovide\s+technical\s+assistance|brokering\s+services\b", re.I)),
    ("ANCILLARY", "financing or financial assistance",
     re.compile(r"\bprovide\s+financing\s+or\s+financial\s+assistance\b", re.I)),
    ("ANCILLARY", "insurance and reinsurance",
     re.compile(r"\bprovide\s+insurance\s+and\s+reinsurance\b", re.I)),
    ("ANCILLARY", "to transport",
     # [2026-05-20] Slack widened from 200 to 300 chars. Audit surfaced
     # Article 3n ¶4 ("to transport... to third countries, crude oil
     # falling under CN code 2709 00, as of 5 December 2022, or petroleum
     # products falling under CN code 2710, as of 5 February 2023, as
     # listed in Annex XXV, which originate in Russia") where the
     # distance from "transport" to "Russia" is 255 chars.
     re.compile(r"\bto\s+transport\b[^.]{0,300}\bRussia\b", re.I)),
    ("ANCILLARY", "to trade, broker or transport",
     re.compile(r"\bto\s+trade,?\s*broker\s*or\s*transport\b", re.I)),

    # ---- TRANSACTION_BAN (entity-level prohibitions, Article 5a etc.)
    ("TRANSACTION_BAN", "engage in any transaction with",
     re.compile(r"\bengage\b[^.]{0,50}\bin\s+any\s+transaction\b", re.I)),
]

# Mapping from direction tag to whether the direction belongs in the
# eventual trade panel. EXPORT and IMPORT are outright goods bans on
# trade flows, so they are trade-relevant. The others are documented but
# excluded from the main panel by default (downstream code can flip these
# flags if desired).
TRADE_RELEVANT: dict[str, bool] = {
    "EXPORT":                            True,
    "IMPORT":                            True,
    "TRANSIT_BAN":                       False,  # transit isn't bilateral EU↔Russia
    "AUTHORISATION_REQUIRED_EXPORT":     False,
    "AUTHORISATION_REQUIRED_IMPORT":     False,
    "AUTHORISATION_REQUIRED_ANCILLARY":  False,
    "PRICE_CAP":                         False,
    "NO_RE_EXPORT":                      False,
    "ANCILLARY":                         False,
    "TRANSACTION_BAN":                   False,
    "OTHER":                             False,
}


# =============================================================================
# ROLE CLASSIFICATION
# =============================================================================
#
# Two-layer architecture:
#   - "Role" classifies the paragraph's LEGAL GRAMMAR (is it a ban, an
#     obligation, an authorisation regime, a derogation?). This is set
#     by `classify_paragraph_role()`.
#   - "Direction" classifies WHICH TRADE FLOW the paragraph affects.
#     This is set by `detect_directions()`.
#
# The two are orthogonal: a NO_RE_EXPORT paragraph has role=obligation,
# an AUTHORISATION_REQUIRED_EXPORT paragraph has role=authorisation_required.
#
# The role classifier checks triggers in this order:
#   1. Derogation patterns  → role=derogation (paragraph skipped downstream)
#   2. Prohibition pattern  → role=prohibition
#   3. Obligation patterns  → role=obligation
#   4. Authorisation patterns → role=authorisation_required
#   5. Anything else → role=other (paragraph skipped downstream)
#
# Derogations are checked FIRST because they sometimes contain the word
# "prohibited" in negating clauses (e.g. "The prohibitions in paragraph 1
# shall not apply to..."). Without this ordering, derogations would
# spuriously classify as prohibitions.
# -----------------------------------------------------------------------------

PROHIBITION_TRIGGER_LABEL: str = "shall be prohibited / is prohibited"
PROHIBITION_RE: re.Pattern = re.compile(r"\b(shall\s+be\s+prohibited|is\s+prohibited)\b", re.I)

DEROGATION_TRIGGERS: list[tuple[re.Pattern, str]] = [
    (re.compile(r"\bshall\s+not\s+apply\b", re.I),                              "shall not apply"),
    (re.compile(r"\bBy\s+way\s+of\s+derogation\b", re.I),                       "By way of derogation"),
    (re.compile(r"\bcompetent\s+authorit(y|ies)\s+may\s+authorise\b", re.I),    "competent authorities may authorise"),
    (re.compile(r"\bThe\s+prohibitions?\s+(in|referred\s+to\s+in)\s+paragraphs?\b", re.I),
                                                                                "The prohibitions in paragraphs"),
    (re.compile(r"\bshall\s+not\s+grant\s+(?:an?\s+|any\s+)?authorisation\b", re.I),
                                                                                  "shall not grant authorisation"),
]

# Set-aside helper: extract the paragraph numbers a derogation derogates FROM.
# Matches "paragraph 1", "paragraphs 1 and 2", "paragraph 1a", "paragraphs 1, 1a and 2".
# Returns the captured numbers as a comma-separated string, empty if not found.
DEROGATED_FROM_RE: re.Pattern = re.compile(
    r"\bparagraphs?\s+"
    r"(\d+[a-z]*(?:\s*(?:,|and)\s*\d+[a-z]*)*)",
    re.I,
)
_PARA_NUM_RE: re.Pattern = re.compile(r"\d+[a-z]*", re.I)

def derogated_from_paragraphs(text: str) -> str:
    """Extract paragraph numbers a derogation cue text refers to.

    Example: "the prohibitions in paragraphs 1 and 2 shall not apply" -> "1, 2"
    Returns an empty string when no paragraph numbers can be extracted.
    """
    m = DEROGATED_FROM_RE.search(text)
    if not m:
        return ""
    nums = _PARA_NUM_RE.findall(m.group(1))
    return ", ".join(nums)


OBLIGATION_TRIGGERS: list[tuple[re.Pattern, str]] = [
    (re.compile(r"\bexporters\s+shall\b", re.I),                                       "exporters shall"),
    (re.compile(r"\bnatural\s+and\s+legal\s+persons[^.]{0,80}\bshall\s+ensure\b", re.I),
                                                                                       "natural and legal persons ... shall ensure"),
    (re.compile(r"\bshall\b[^.]{0,30}\bcontractually\s+prohibit\b", re.I),             "shall contractually prohibit"),
]

AUTHORISATION_TRIGGERS: list[tuple[re.Pattern, str]] = [
    (re.compile(r"\bprior\s+authorisation\s+shall\s+be\s+required\b", re.I),
                                                              "prior authorisation shall be required"),
    (re.compile(r"\bauthorisation\s+shall\s+be\s+required\b", re.I),
                                                              "authorisation shall be required"),
    (re.compile(r"\bshall\s+be\s+subject\s+to\s+(an\s+)?authorisation\b", re.I),
                                                              "shall be subject to authorisation"),
]


def classify_paragraph_role(text: str) -> tuple[str, str]:
    """Return (role, role_trigger_phrase) for a paragraph.

    The role determines whether the paragraph contributes to the output.
    Derogation and 'other' paragraphs are skipped downstream.
    """
    for pattern, trigger in DEROGATION_TRIGGERS:
        if pattern.search(text):
            return "derogation", trigger
    if PROHIBITION_RE.search(text):
        return "prohibition", PROHIBITION_TRIGGER_LABEL
    for pattern, trigger in OBLIGATION_TRIGGERS:
        if pattern.search(text):
            return "obligation", trigger
    for pattern, trigger in AUTHORISATION_TRIGGERS:
        if pattern.search(text):
            return "authorisation_required", trigger
    return "other", ""


# =============================================================================
# ANNEX-CITATION EXTRACTION
# =============================================================================
#
# Three subtleties handled here:
#
# 1. Plural form: "Annexes X, Y and Z" doesn't match a singular "Annex X"
#    regex. We pre-process the text to rewrite plurals as comma-separated
#    singulars.
#
# 2. External references: "Annex X to Regulation (EU) No N/Y" or
#    "Annex X to Decision N/N/CFSP" refers to an annex of ANOTHER
#    regulation, not Reg. 833/2014. We extract these separately and mask
#    them before running the internal-annex regex.
#
# 3. The "NO_INTERNAL_ANNEX" sentinel: when a paragraph cites only external
#    annexes (typical for the firearms ban in Article 2aa, which cites only
#    Annex I of Reg. 258/2012), we still emit a row with annex=
#    NO_INTERNAL_ANNEX so the external citation remains visible.
# -----------------------------------------------------------------------------

# Internal annex citations: "Annex X", where X is a Roman numeral optionally
# followed by a single uppercase letter (e.g. XXXVIIIA).
ANNEX_RE: re.Pattern = re.compile(r"\bAnnex\s+([IVXLC]+[A-Z]?)\b", re.I)

# Plural form: "Annexes X, Y and Z", "Annexes X and Y", etc.
PLURAL_ANNEX_RE: re.Pattern = re.compile(
    r"\bAnnexes\s+("
    r"[IVXLC]+[A-Z]?"
    r"(?:\s*,\s*[IVXLC]+[A-Z]?)*"
    r"(?:\s+and\s+[IVXLC]+[A-Z]?)?"
    r")\b",
    re.I,
)

# External references: "Annex X to Regulation (EU/EC) No N/Y" or
# "Annex X to Decision N/N/CFSP".
# Does NOT match "Annex X to this Regulation" — that stays internal.
EXTERNAL_REF_RE: re.Pattern = re.compile(
    r"\bAnnex\s+([IVXLC]+[A-Z]?)\s+to\s+"
    r"("
    r"Regulation\s+\(EU\)\s+(?:No\s+)?[\d/]+"
    r"|Regulation\s+\(EC\)\s+(?:No\s+)?[\d/]+"
    r"|Decision\s+[\d/]+(?:/[A-Z]+)?"
    r")",
    re.I,
)


def expand_plural_annex_refs(text: str) -> str:
    """Rewrite "Annexes X, Y and Z" as "Annex X, Annex Y, Annex Z".

    The internal-annex regex matches only the singular form ("Annex N"),
    so without this rewrite plural references would be missed entirely.
    """
    def repl(match: re.Match) -> str:
        numerals = re.findall(r"[IVXLC]+[A-Z]?", match.group(1), re.I)
        return ", ".join(f"Annex {n.upper()}" for n in numerals)

    return PLURAL_ANNEX_RE.sub(repl, text)


def extract_external_refs(text: str) -> list[str]:
    """Return human-readable strings like 'Annex I to Regulation (EU) No 258/2012'."""
    return [
        f"Annex {numeral.upper()} to {source}"
        for numeral, source in EXTERNAL_REF_RE.findall(text)
    ]


def mask_external_refs(text: str) -> str:
    """Replace external-ref substrings with a placeholder.

    This is run before extract_annexes() so the internal-annex regex
    doesn't capture annexes of other regulations.
    """
    return EXTERNAL_REF_RE.sub("[EXTERNAL_REF]", text)


def extract_annexes(text: str) -> list[str]:
    """Return sorted, deduplicated list of annex labels like 'ANNEX II'.

    The caller should mask external refs first; otherwise the result
    will include annexes of other regulations.
    """
    return sorted({"ANNEX " + a.upper() for a in ANNEX_RE.findall(text)})


# =============================================================================
# DIRECTION EXTRACTION
# =============================================================================

@dataclass
class DirectionHit:
    """One direction-pattern match within a paragraph."""

    direction: str        # e.g. "EXPORT"
    pattern_label: str    # human-readable description of which regex matched
    evidence_snippet: str # short text snippet (~150 chars) showing the match


def detect_directions(text: str) -> list[DirectionHit]:
    """Try every DIRECTION_PATTERNS regex on the text.

    Returns one DirectionHit per matching pattern. A single paragraph
    can match multiple patterns — each match produces a separate hit.
    """
    hits: list[DirectionHit] = []
    for direction, label, pattern in DIRECTION_PATTERNS:
        match = pattern.search(text)
        if match is None:
            continue

        # Snippet: ~40 chars before and ~120 chars after the match.
        # Cap and clean whitespace for clean CSV cells.
        start = max(0, match.start() - 40)
        end = min(len(text), match.end() + 120)
        snippet = text[start:end].strip()
        # Drop a possible leading partial word if we cut mid-word
        if start > 0:
            snippet = re.sub(r"^\S*\s", "", snippet, count=1)
        snippet = re.sub(r"\s+", " ", snippet)
        hits.append(DirectionHit(direction, label, snippet))

    return hits


# =============================================================================
# HTML → PLAIN TEXT → ARTICLES → PARAGRAPHS
# =============================================================================
#
# Two critical regex notes:
#
# 1. ARTICLE_HEADER_RE uses [a-z]* (zero-or-more), NOT [a-z]?
#    EU regulations insert new articles using letter suffixes (3a, 3b, ...).
#    Some have two-letter suffixes (2aa, 5aa, 12ga, 12gb). With [a-z]?
#    "Article 2aa" silently merges into "Article 2a" because the regex
#    can match at most one trailing letter.
#
# 2. split_into_articles() truncates at the first ANNEX header.
#    Article 14 (final-provisions) is the last article. After it the
#    document continues with annex tables. Without truncation, all annex
#    content gets absorbed into Article 14, and the numbered annex
#    entries are misread as paragraphs.
# -----------------------------------------------------------------------------

ARTICLE_HEADER_RE: re.Pattern = re.compile(r"(?m)^\s*Article\s+(\d+[a-z]*)\s*$")
ANNEX_START_RE:    re.Pattern = re.compile(r"(?m)^\s*ANNEX\s+[IVXLC]+[A-Z]?\s*$")
PARA_HEADER_RE:    re.Pattern = re.compile(r"(?m)^\s*(\d+[a-z]*)\.\s")


def html_to_plain(html: str) -> str:
    """Convert HTML to plain text, preserving line breaks at element boundaries."""
    soup = BeautifulSoup(html, "lxml")
    text = soup.get_text("\n")
    # [2026-05-20] Strip EUR-Lex amendment markers (▼M44, ▼M57, ▼M65...).
    # They sit inline with adjacent text and reduce readability of the
    # `evidence_for_direction` column. Companion to the lettered-paragraph
    # fix above: removing them helps ensure paragraph headers like "1a."
    # start cleanly at column 0 after whitespace normalisation.
    text = re.sub(r"▼M\d+", "\n", text)
    # Normalise whitespace: drop trailing spaces, collapse multi-newlines.
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def split_into_articles(text: str) -> list[tuple[str, str]]:
    """Split the document into (article_number, article_body) tuples.

    The "article number" is a string like "3", "3a", "12g", "2aa".
    The body is the raw text between this article's header and the next.
    """
    # Truncate at first ANNEX header so the last article doesn't absorb
    # the annex tables.
    annex_start = ANNEX_START_RE.search(text)
    articles_text = text[:annex_start.start()] if annex_start else text

    matches = list(ARTICLE_HEADER_RE.finditer(articles_text))
    out: list[tuple[str, str]] = []
    for i, match in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(articles_text)
        out.append((match.group(1), articles_text[match.end():end].strip()))
    return out


def split_into_paragraphs(article_text: str) -> list[tuple[str, str]]:
    """Split an article body into (paragraph_number, paragraph_body) tuples.

    Paragraphs are numbered "1.", "2.", etc. at line start. Articles
    without numbered paragraphs return as a single ("0", whole_body) entry.
    """
    matches = list(PARA_HEADER_RE.finditer(article_text))
    if not matches:
        return [("0", article_text)] if article_text.strip() else []

    out: list[tuple[str, str]] = []
    for i, match in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(article_text)
        out.append((match.group(1), article_text[match.end():end].strip()))
    return out


def clean_cell_text(text: str) -> str:
    """Aggressive whitespace cleanup so the cell is safe for CSV output.

    Newlines inside cells confuse Excel; collapsing them prevents
    spurious row breaks when the CSV is opened.
    """
    text = re.sub(r"[\r\n\u2028\u2029]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# =============================================================================
# MAIN EXTRACTION LOOP
# =============================================================================

def build_row(
    law_version: str,
    article: str,
    paragraph: str,
    annex: str,
    direction: str,
    matched_pattern: str,
    evidence: str,
    role: str,
    role_trigger: str,
    external_refs_str: str,
    full_paragraph: str,
) -> dict:
    """Assemble one output row with all 12 schema columns."""
    return {
        "law_version":                   law_version,
        "article":                       article,
        "paragraph":                     paragraph,
        "annex":                         annex,
        "direction":                     direction,
        "trade_relevant":                TRADE_RELEVANT[direction],
        "role":                          role,
        "role_trigger":                  role_trigger,
        "matched_pattern":               matched_pattern,
        "evidence_for_direction":        evidence,
        "annexes_of_other_regulations":  external_refs_str,
        "full_paragraph":                full_paragraph,
    }


def build_setaside_row(
    law_version: str,
    article: str,
    paragraph: str,
    annex: str,
    role: str,
    role_trigger: str,
    direction: str,
    derogated_from: str,
    paragraph_snippet: str,
) -> dict:
    """Assemble one set-aside CSV row.

    Set-aside rows record annex citations in paragraphs the classifier does
    NOT emit as EXPORT or IMPORT (derogation, authorisation_required,
    obligation-only, OTHER, plus non-trade directions such as TRANSIT_BAN,
    NO_RE_EXPORT, PRICE_CAP, TRANSACTION_BAN, ANCILLARY, AUTHORISATION_REQUIRED_*).
    The downstream directional Step 3 EXCLUDES these annexes by default
    (set-aside principle). See companion docs for refinement workflow.
    """
    return {
        "law_version":                   law_version,
        "article":                       article,
        "paragraph":                     paragraph,
        "annex":                         annex,
        "role":                          role,
        "role_trigger":                  role_trigger,
        "direction":                     direction,
        "derogated_from_paragraphs":     derogated_from,
        "matched_text":                  role_trigger,
        "paragraph_snippet":             paragraph_snippet,
    }


def _make_setaside_snippet(paragraph_body: str, annex: str) -> str:
    """Up to SETASIDE_SNIPPET_WIDTH chars of the paragraph, starting near the
    annex citation if possible. Newlines collapsed (matches clean_cell_text).
    """
    cleaned = clean_cell_text(paragraph_body)
    # try to anchor the snippet around the literal annex token
    needle = f"Annex {annex}"
    i = cleaned.lower().find(needle.lower())
    if i < 0:
        return cleaned[:SETASIDE_SNIPPET_WIDTH]
    start = max(0, i - 20)
    return cleaned[start:start + SETASIDE_SNIPPET_WIDTH]


def process_version(celex_id: str) -> tuple[list[dict], list[dict], list[str], int]:
    """Run the full pipeline on one law version.

    Returns:
        rows         - one dict per directions-CSV row
        setaside     - one dict per set-aside-CSV row (NEW 2026-05-28)
        article_ids  - list of detected article identifiers (for diagnostics)
        n_chars      - plain-text length (for diagnostics)

    Set-aside rows record INTERNAL annex citations in paragraphs that do NOT
    contribute an EXPORT or IMPORT direction (derogation, authorisation_required,
    obligation, OTHER, plus non-trade directions). See the trio's TECHNICAL §11
    (D.21-D.24) for the design rationale.
    """
    html = get_html_by_celex_id(celex_id)
    text = html_to_plain(html)
    articles = split_into_articles(text)

    rows: list[dict] = []
    setaside: list[dict] = []
    TRADE_DIRS = {"EXPORT", "IMPORT"}

    for article_number, article_body in articles:
        for paragraph_number, paragraph_body in split_into_paragraphs(article_body):
            # ---- Layer 1: role classification ---------------------------
            role, role_trigger = classify_paragraph_role(paragraph_body)

            # ---- Annex extraction is needed for BOTH the kept branches
            # (prohibition / obligation / authorisation_required) AND the
            # derogation branch (for set-aside emission). The `other` branch
            # is also routed here so any annex citation it contains lands in
            # set-aside.
            expanded_text     = expand_plural_annex_refs(paragraph_body)
            external_refs     = extract_external_refs(expanded_text)
            masked_text       = mask_external_refs(expanded_text)
            internal_annexes  = extract_annexes(masked_text)
            external_refs_str = "; ".join(external_refs)

            # ===== SET-ASIDE: derogation branch ==========================
            # Derogations were previously dropped entirely. Now we emit a
            # set-aside row per INTERNAL annex citation, with a snippet and
            # the paragraph numbers being derogated from when extractable.
            if role == "derogation":
                if internal_annexes:
                    derog_from = derogated_from_paragraphs(role_trigger + " " + paragraph_body[:300])
                    for annex in internal_annexes:
                        setaside.append(build_setaside_row(
                            law_version=celex_id,
                            article=article_number,
                            paragraph=paragraph_number,
                            annex=annex,
                            role=role,
                            role_trigger=role_trigger,
                            direction="",
                            derogated_from=derog_from,
                            paragraph_snippet=_make_setaside_snippet(paragraph_body, annex),
                        ))
                continue

            if role == "other":
                # Some `other`-tagged paragraphs cite annexes (e.g. broadcasting
                # bans, scope clauses). Capture their internal-annex citations.
                if internal_annexes:
                    for annex in internal_annexes:
                        setaside.append(build_setaside_row(
                            law_version=celex_id,
                            article=article_number,
                            paragraph=paragraph_number,
                            annex=annex,
                            role=role,
                            role_trigger="",
                            direction="",
                            derogated_from="",
                            paragraph_snippet=_make_setaside_snippet(paragraph_body, annex),
                        ))
                continue

            # ---- From here on: role is prohibition / obligation /
            #      authorisation_required. Existing emit logic preserved.

            if not internal_annexes and not external_refs:
                continue

            direction_hits = detect_directions(paragraph_body)
            full_paragraph = clean_cell_text(paragraph_body)
            annexes_to_emit = internal_annexes or [NO_INTERNAL_ANNEX_SENTINEL]

            # Case A: no direction pattern matched -> OTHER row per annex.
            if not direction_hits:
                for annex in annexes_to_emit:
                    rows.append(build_row(
                        celex_id, article_number, paragraph_number,
                        annex, "OTHER",
                        matched_pattern="",
                        evidence="",
                        role=role,
                        role_trigger=role_trigger,
                        external_refs_str=external_refs_str,
                        full_paragraph=full_paragraph,
                    ))
                # Set-aside: same internal annexes also recorded as set-aside,
                # because this paragraph contributes NO trade direction. We
                # skip the NO_INTERNAL_ANNEX sentinel (out of scope by design).
                for annex in internal_annexes:
                    setaside.append(build_setaside_row(
                        law_version=celex_id,
                        article=article_number,
                        paragraph=paragraph_number,
                        annex=annex,
                        role=role,
                        role_trigger=role_trigger,
                        direction="OTHER",
                        derogated_from="",
                        paragraph_snippet=_make_setaside_snippet(paragraph_body, annex),
                    ))
                continue

            # Case B: one or more directions matched. Cartesian product:
            # one row per (annex × direction) in the directions CSV.
            for annex in annexes_to_emit:
                for hit in direction_hits:
                    rows.append(build_row(
                        celex_id, article_number, paragraph_number,
                        annex, hit.direction,
                        matched_pattern=hit.pattern_label,
                        evidence=hit.evidence_snippet,
                        role=role,
                        role_trigger=role_trigger,
                        external_refs_str=external_refs_str,
                        full_paragraph=full_paragraph,
                    ))

            # Set-aside complement: if the paragraph contributed NO trade
            # direction (every hit is non-trade), record its INTERNAL annex
            # citations to set-aside as well. If it contributed at least one
            # trade direction, the annex is directionally classified and
            # set-aside emission would be redundant noise (per D.22).
            trade_in_hits = {h.direction for h in direction_hits} & TRADE_DIRS
            if not trade_in_hits:
                non_trade = sorted({h.direction for h in direction_hits})
                for annex in internal_annexes:
                    setaside.append(build_setaside_row(
                        law_version=celex_id,
                        article=article_number,
                        paragraph=paragraph_number,
                        annex=annex,
                        role=role,
                        role_trigger=role_trigger,
                        direction=";".join(non_trade),
                        derogated_from="",
                        paragraph_snippet=_make_setaside_snippet(paragraph_body, annex),
                    ))

    article_ids = [a[0] for a in articles]
    return rows, setaside, article_ids, len(text)


# =============================================================================
# DIAGNOSTIC REPORT
# =============================================================================

def print_diagnostics(df: pd.DataFrame) -> None:
    """Print summary tables to stdout for sanity-checking the output.

    Uses .reset_index() rather than .unstack() to avoid a known pandas
    incompatibility on some Colab installations.
    """
    print("\nRow counts per version:")
    print(df.groupby("law_version").size().to_string())

    print("\nDirection counts (totals):")
    print(df["direction"].value_counts().to_string())

    print("\nRole counts per version:")
    print(df.groupby(["law_version", "role"])
          .size().reset_index(name="n")
          .to_string(index=False))

    print("\nDirection counts per version:")
    print(df.groupby(["law_version", "direction"])
          .size().reset_index(name="n")
          .to_string(index=False))

    print("\nUnique annexes per version (trade-relevant only):")
    trade_only = df[df["trade_relevant"]]
    print(trade_only.groupby("law_version")["annex"].nunique().to_string())

    # For each direction, which articles use it? Useful for spotting
    # false positives.
    print("\nDirection × article (across all versions):")
    direction_order = [
        "EXPORT", "IMPORT", "TRANSIT_BAN",
        "AUTHORISATION_REQUIRED_EXPORT",
        "AUTHORISATION_REQUIRED_IMPORT",
        "AUTHORISATION_REQUIRED_ANCILLARY",
        "PRICE_CAP", "NO_RE_EXPORT",
        "ANCILLARY", "TRANSACTION_BAN", "OTHER",
    ]
    for direction in direction_order:
        arts = sorted(df[df["direction"] == direction]["article"].unique())
        if arts:
            print(f"  {direction:34s}: {', '.join(arts)}")


# =============================================================================
# ENTRY POINT
# =============================================================================

def main(versions: Iterable[str] = ALL_VERSIONS,
         output_csv: str = OUTPUT_CSV,
         setaside_csv: str = SETASIDE_OUTPUT_CSV) -> pd.DataFrame:
    """Run the parser on a list of CELEX versions and write the two CSVs.

    Returns the assembled directions-CSV DataFrame so the function is also
    usable programmatically. The set-aside DataFrame is written to disk
    but not returned.
    """
    all_rows: list[dict] = []
    all_setaside: list[dict] = []
    versions = list(versions)
    print(f"Processing {len(versions)} versions ...\n")
    t_start = time.time()

    for i, version in enumerate(versions, start=1):
        t0 = time.time()
        try:
            rows, setaside, article_ids, n_chars = process_version(version)
            all_rows.extend(rows)
            all_setaside.extend(setaside)
            print(
                f"  [{i:2d}/{len(versions)}] {version}: "
                f"{len(article_ids):3d} articles, "
                f"{n_chars/1000:6.0f}k chars, "
                f"{len(rows):3d} dir-rows, "
                f"{len(setaside):3d} set-aside ({time.time()-t0:.1f}s)"
            )
        except Exception as exc:
            print(f"  [{i:2d}/{len(versions)}] {version}: "
                  f"FAILED — {type(exc).__name__}: {exc}")

    print(f"\nDone in {time.time()-t_start:.1f}s. Building outputs ...")

    # ---- Directions CSV (UNCHANGED schema and ordering) --------------------
    df = pd.DataFrame(all_rows)
    df = df[[
        "law_version", "article", "paragraph", "annex",
        "direction", "trade_relevant",
        "role", "role_trigger",
        "matched_pattern", "evidence_for_direction",
        "annexes_of_other_regulations",
        "full_paragraph",
    ]]
    df = df.sort_values(["law_version", "article", "paragraph", "annex", "direction"])
    df.to_csv(output_csv, index=False)

    # ---- Set-aside CSV (NEW 2026-05-28) ------------------------------------
    sa = pd.DataFrame(all_setaside)
    if len(sa):
        sa = sa[[
            "law_version", "article", "paragraph", "annex",
            "role", "role_trigger",
            "direction", "derogated_from_paragraphs",
            "matched_text", "paragraph_snippet",
        ]]
        sa = sa.sort_values(["law_version", "article", "paragraph", "annex"])
    sa.to_csv(setaside_csv, index=False)

    print(f"\n{'=' * 70}")
    print(f"Directions: {len(df):5d} rows ({df['law_version'].nunique()} versions) "
          f"-> {output_csv}")
    print(f"Set-aside:  {len(sa):5d} rows -> {setaside_csv}")
    print(f"{'=' * 70}")

    print_diagnostics(df)
    return df


if __name__ == "__main__":
    main()

Processing 34 versions ...

  [ 1/34] 32014R0833:  14 articles,     27k chars,   4 dir-rows,  10 set-aside (1.6s)
  [ 2/34] 02014R0833-20140912:  16 articles,     34k chars,  10 dir-rows,  17 set-aside (1.3s)
  [ 3/34] 02014R0833-20141206:  16 articles,     37k chars,   9 dir-rows,  17 set-aside (1.3s)
  [ 4/34] 02014R0833-20151009:  16 articles,     39k chars,   9 dir-rows,  17 set-aside (1.4s)
  [ 5/34] 02014R0833-20171202:  16 articles,     37k chars,   9 dir-rows,  17 set-aside (1.3s)
  [ 6/34] 02014R0833-20190709:  16 articles,     37k chars,   9 dir-rows,  17 set-aside (1.3s)
  [ 7/34] 02014R0833-20220228:  32 articles,    147k chars,  13 dir-rows,  32 set-aside (2.5s)
  [ 8/34] 02014R0833-20220316:  40 articles,    206k chars,  22 dir-rows,  31 set-aside (2.7s)
  [ 9/34] 02014R0833-20220413:  48 articles,    333k chars,  26 dir-rows,  36 set-aside (2.9s)
  [10/34] 02014R0833-20220604:  51 articles,    356k chars,  29 dir-rows,  43 set-aside (3.1s)
  [11/34] 02014R0833-20220722: 

In [ ]:
# ============================================================
# DISCOVERY AUDIT — surface OTHER-direction rows for review
# ============================================================
# Reads the parser CSV produced by the cell above and inspects every row
# classified as direction=OTHER, in two stages of decreasing priority:
#
#   Stage 1: role=authorisation_required + direction=OTHER
#            (auth phrase detected but no direction pattern fired —
#            almost always a real gap)
#
#   Stage 2: role=prohibition/obligation + direction=OTHER
#            (deduplicated by paragraph text). If RU-STEP2_hs6_long.csv
#            is available, this is split into:
#              Stage 2a — annexes containing goods CN codes
#                         (high priority — review carefully)
#              Stage 2b — annexes without (asset-freeze / name-list
#                         annexes — OTHER is usually correct)
#
# Also writes discovery_audit_russia.csv with one row per dedup group
# in the same folder as the parser CSV.
# ============================================================

# ── Paths (edit if running standalone) ──────────────────────────────
PARSER_CSV = OUTPUT_CSV                            # set by the parser cell above
STEP2_CSV  = "/content/RU-STEP2_hs6_long.csv"      # set to None if not uploaded
# ────────────────────────────────────────────────────────────────────


import sys
from pathlib import Path

import pandas as pd


SNIPPET_LEN: int = 500


def stage1_authorisation_misses(df: pd.DataFrame) -> pd.DataFrame:
    """Rows where role=authorisation_required but no direction fired."""
    return df[
        (df["role"] == "authorisation_required")
        & (df["direction"] == "OTHER")
    ].copy()


def stage2_prohibition_misses(df: pd.DataFrame) -> pd.DataFrame:
    """Rows where role is prohibition/obligation but direction=OTHER."""
    return df[
        df["role"].isin(["prohibition", "obligation"])
        & (df["direction"] == "OTHER")
    ].copy()


def dedupe_by_paragraph(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse rows sharing the same full_paragraph text.

    Adds n_versions (how many versions this paragraph appeared in) and
    sorts by article + paragraph for readability.
    """
    grouped = (
        df.groupby(["article", "paragraph", "annex", "role", "full_paragraph"],
                   as_index=False, dropna=False)
          .agg(n_versions=("law_version", "nunique"),
               first_version=("law_version", "min"),
               last_version=("law_version", "max"))
    )
    return grouped.sort_values(["article", "paragraph", "annex"])


def annexes_with_cn_codes(step2_path: Path) -> set[str]:
    """Read RU-STEP2_hs6_long.csv and return the set of annex labels."""
    step2 = pd.read_csv(step2_path)
    annex_col = next((c for c in step2.columns
                      if c.lower() in ("annex", "annex_name", "annex_label")), None)
    if annex_col is None:
        print(f"Warning: no 'annex' column in {step2_path.name}; "
              f"skipping CN-annex cross-reference.")
        return set()
    return set(step2[annex_col].dropna().astype(str).str.upper().unique())


def print_paragraph(row: pd.Series) -> None:
    """Pretty-print one suspicious paragraph for human review."""
    label = (
        f"Article {row['article']:>4} ¶{row['paragraph']:<3} {row['annex']:<14} "
        f"role={row['role']:<23}"
    )
    if "n_versions" in row.index:
        label += f" appears in {row['n_versions']} version(s)"
    print(f"\n  {label}")
    text = str(row["full_paragraph"])
    snippet = text[:SNIPPET_LEN]
    if len(text) > SNIPPET_LEN:
        snippet += " ..."
    print(f"    {snippet}")


def print_stage(title: str, df: pd.DataFrame) -> None:
    print()
    print("=" * 75)
    print(title)
    print("=" * 75)
    print(f"\n{len(df)} rows.")
    if len(df) == 0:
        return
    print("\nPer-article counts:")
    print(df.groupby(["article", "paragraph"], dropna=False)
          .size().reset_index(name="n").to_string(index=False))
    print("\nFull text of each unique paragraph:")
    for _, row in df.iterrows():
        print_paragraph(row)


def main(argv: list[str]) -> None:
    if len(argv) < 2:
        print(__doc__)
        sys.exit(1)

    csv_path = Path(argv[1])
    if not csv_path.exists():
        print(f"Error: {csv_path} does not exist.")
        sys.exit(1)

    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows across {df['law_version'].nunique()} "
          f"versions from {csv_path.name}")

    # --- Stage 1: authorisation_required + OTHER ---
    stage1 = stage1_authorisation_misses(df)
    stage1_dedup = dedupe_by_paragraph(stage1)
    print_stage(
        "STAGE 1 — role=authorisation_required, direction=OTHER  (HIGHEST PRIORITY)",
        stage1_dedup,
    )

    # --- Stage 2: prohibition/obligation + OTHER ---
    stage2 = stage2_prohibition_misses(df)
    stage2_dedup = dedupe_by_paragraph(stage2)

    if len(argv) >= 3:
        step2_path = Path(argv[2])
        if step2_path.exists():
            cn_annexes = annexes_with_cn_codes(step2_path)
            print(f"\nCross-reference: {len(cn_annexes)} annex labels found in "
                  f"{step2_path.name}.")
            mask = stage2_dedup["annex"].str.upper().isin(cn_annexes)
            stage2_priority = stage2_dedup[mask]
            stage2_lowpri = stage2_dedup[~mask]
            print_stage(
                "STAGE 2a — role=prohibition/obligation, direction=OTHER, "
                "annex CONTAINS CN codes  (review carefully)",
                stage2_priority,
            )
            print_stage(
                "STAGE 2b — role=prohibition/obligation, direction=OTHER, "
                "annex has no CN codes  (usually correct OTHER, skim only)",
                stage2_lowpri,
            )
        else:
            print(f"Warning: {step2_path} does not exist; running Stage 2 "
                  f"without CN-annex cross-reference.")
            print_stage(
                "STAGE 2 — role=prohibition/obligation, direction=OTHER",
                stage2_dedup,
            )
    else:
        print_stage(
            "STAGE 2 — role=prohibition/obligation, direction=OTHER  "
            "(no _STEP2 cross-reference supplied)",
            stage2_dedup,
        )

    # --- Write a spreadsheet-friendly dump for triage ---
    out_path = csv_path.with_name("discovery_audit_russia.csv")
    combined = pd.concat([
        stage1_dedup.assign(stage="1_authorisation_required"),
        stage2_dedup.assign(stage="2_prohibition_or_obligation"),
    ], ignore_index=True)
    combined.to_csv(out_path, index=False)
    print(f"\nWrote {len(combined)} rows to {out_path}")



# ── Run ────────────────────────────────────────────────────────────
_argv = ["audit_cell", PARSER_CSV]
if STEP2_CSV is not None:
    _argv.append(STEP2_CSV)
main(_argv)

Loaded 1619 rows across 34 versions from RU-annex_directions_all_versions_2026-06-01_14-50.csv

STAGE 1 — role=authorisation_required, direction=OTHER  (HIGHEST PRIORITY)

0 rows.

STAGE 2 — role=prohibition/obligation, direction=OTHER

24 rows.

Per-article counts:
article paragraph  n
     11         1  1
    12f         3  1
   12ga         1  1
     2f         1  1
     2f         3  1
     3c         3  1
     3d        1b  1
    3nb         1  3
      5         0  1
      5         1  2
      5         2  3
      5         3  3
      5         4  1
    5ah         1  1
    5ah         2  1
     5h         0  1
     5h         1  1

Full text of each unique paragraph:

  Article   11 ¶1   ANNEX VIII     role=prohibition             appears in 1 version(s)
    No claims in connection with any contract or transaction the performance of which has been affected, directly or indirectly, in whole or in part, by the measures imposed under this Regulation, including claims for indemnity o